# SeDriCa Perception · Q2(b) — Baseline rule

Inputs: `red_score`, `person_score`, `v2x_light`. Not used: `v2x_sample_time_s` (part (c)) and `crossing_reference.csv`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import cv2 as cv          # not needed yet: the baseline uses only the supplied scores
import pandas as pd       # extra: just to read the CSV

## Thresholds
Edit these and re-run.

| name | value | used for |
|---|---|---|
| `thv1` | 0.7 | V2X says **red**: `red_score` above this → camera agrees → STOP |
| `thv2` | 0.3 | V2X says **not red**: `red_score` below this → camera agrees → GO |
| `thv3` | 0.5 | `person_score` above this → a person is probably there → STOP |

Between `thv2` and `thv1` the camera is unclear, so a disagreement with V2X stays unresolved.

In [2]:
thv1 = 0.7
thv2 = 0.3
thv3 = 0.5

## What each state means
- **GO** — keep going at the optimal speed.
- **SLOW** (the "GO SLOW" state) — reduce speed to avoid any accident, so the car can easily stop.
- **STOP** — complete halt (speed = 0).

## The rule
The more cautious of the two checks wins (STOP > SLOW > GO); both reasons are kept.
- **Light:** V2X red → STOP if `red_score > thv1`, else SLOW. Not red → GO if `red_score < thv2`, else SLOW.
- **Person:** `person_score > thv3` → STOP, else GO. This also stops for a person outside the road.

In [3]:
RANK = {"GO": 0, "SLOW": 1, "STOP": 2}

def light_check(v2x_light, red_score):
    if v2x_light == "red":
        if red_score > thv1:
            return "STOP", "the evidence very highly suggests that the signal is red"
        return "SLOW", "there is contradictory evidence between the light signal and the red_score"
    if red_score < thv2:
        return "GO", "the evidence very highly suggests that the signal is green"
    return "SLOW", "there is conflicting evidence between the light signal and the red_score"

def person_check(person_score):
    if person_score > thv3:
        return "STOP", "there is evidence that a person exists"   # CHANGED: was SLOW, a person now means a complete stop
    return "GO", "evidence very lowly suggests that a person exists"

def decide(v2x_light, red_score, person_score):
    """Return (decision, reason). Most cautious of the two checks wins."""
    (d1, r1), (d2, r2) = light_check(v2x_light, red_score), person_check(person_score)
    decision = max(d1, d2, key=RANK.get)
    return decision, f"light: {r1} | person: {r2}"

## Run on all three episodes
`message_age_s` is for reference only; the rule doesn't use it.

In [4]:
ev = pd.read_csv("SeDriCa_perception_starter_data/city/crossing_evidence.csv")
EPISODES = ["clear_green", "late_red", "person_crossing"]

ev[["decision", "reason"]] = [decide(l, r, p) for l, r, p in zip(ev.v2x_light, ev.red_score, ev.person_score)]
ev["message_age_s"] = (ev.time_s - ev.v2x_sample_time_s).round(2)

pd.set_option("display.max_colwidth", None, "display.width", 250)
cols = ["episode", "frame", "time_s", "red_score", "person_score", "v2x_light", "message_age_s", "decision"]
ev.groupby("episode").decision.value_counts().unstack(fill_value=0).reindex(columns=["GO", "SLOW", "STOP"], fill_value=0).loc[EPISODES]

decision,GO,SLOW,STOP
episode,,,
clear_green,20,1,3
late_red,8,4,12
person_crossing,9,0,15


In [5]:
from IPython.display import display, Markdown

show_cols = ["time_s", "red_score", "person_score", "v2x_light", "message_age_s", "decision", "reason"]
for ep in EPISODES:
    display(Markdown(f"### {ep}"))
    display(ev[ev.episode == ep].set_index("frame")[show_cols].style.format({"time_s": "{:.1f}", "red_score": "{:.3f}", "person_score": "{:.3f}", "message_age_s": "{:.1f}"}).set_properties(subset=["reason"], **{"text-align": "left", "white-space": "normal", "min-width": "430px"}))

### clear_green

,time_s,red_score,person_score,v2x_light,message_age_s,decision,reason
frame,,,,,,,
0,0.0,0.010,0.010,green,0.0,GO,light: the evidence very highly suggests that the signal is green | person: evidence very lowly suggests that a person exists
1,0.1,0.010,0.113,green,0.0,GO,light: the evidence very highly suggests that the signal is green | person: evidence very lowly suggests that a person exists
2,0.2,0.028,0.010,green,0.0,GO,light: the evidence very highly suggests that the signal is green | person: evidence very lowly suggests that a person exists
3,0.3,0.010,0.153,green,0.0,GO,light: the evidence very highly suggests that the signal is green | person: evidence very lowly suggests that a person exists
4,0.4,0.040,0.456,green,0.0,GO,light: the evidence very highly suggests that the signal is green | person: evidence very lowly suggests that a person exists
5,0.5,0.010,0.219,green,0.5,GO,light: the evidence very highly suggests that the signal is green | person: evidence very lowly suggests that a person exists
6,0.6,0.365,0.770,green,0.6,STOP,light: there is conflicting evidence between the light signal and the red_score | person: there is evidence that a person exists
7,0.7,0.094,0.770,green,0.0,STOP,light: the evidence very highly suggests that the signal is green | person: there is evidence that a person exists
8,0.8,0.010,0.185,green,0.0,GO,light: the evidence very highly suggests that the signal is green | person: evidence very lowly suggests that a person exists


### late_red

,time_s,red_score,person_score,v2x_light,message_age_s,decision,reason
frame,,,,,,,
0,0.0,0.090,0.146,green,0.0,GO,light: the evidence very highly suggests that the signal is green | person: evidence very lowly suggests that a person exists
1,0.1,0.010,0.541,green,0.0,STOP,light: the evidence very highly suggests that the signal is green | person: there is evidence that a person exists
2,0.2,0.010,0.010,green,0.0,GO,light: the evidence very highly suggests that the signal is green | person: evidence very lowly suggests that a person exists
3,0.3,0.017,0.335,green,0.0,GO,light: the evidence very highly suggests that the signal is green | person: evidence very lowly suggests that a person exists
4,0.4,0.108,0.078,green,0.0,GO,light: the evidence very highly suggests that the signal is green | person: evidence very lowly suggests that a person exists
5,0.5,0.078,0.192,green,0.5,GO,light: the evidence very highly suggests that the signal is green | person: evidence very lowly suggests that a person exists
6,0.6,0.010,0.010,green,0.6,GO,light: the evidence very highly suggests that the signal is green | person: evidence very lowly suggests that a person exists
7,0.7,0.069,0.010,green,0.0,GO,light: the evidence very highly suggests that the signal is green | person: evidence very lowly suggests that a person exists
8,0.8,0.010,0.018,green,0.0,GO,light: the evidence very highly suggests that the signal is green | person: evidence very lowly suggests that a person exists


### person_crossing

,time_s,red_score,person_score,v2x_light,message_age_s,decision,reason
frame,,,,,,,
0,0.0,0.010,0.187,green,0.0,GO,light: the evidence very highly suggests that the signal is green | person: evidence very lowly suggests that a person exists
1,0.1,0.010,0.010,green,0.0,GO,light: the evidence very highly suggests that the signal is green | person: evidence very lowly suggests that a person exists
2,0.2,0.224,0.010,green,0.0,GO,light: the evidence very highly suggests that the signal is green | person: evidence very lowly suggests that a person exists
3,0.3,0.010,0.010,green,0.0,GO,light: the evidence very highly suggests that the signal is green | person: evidence very lowly suggests that a person exists
4,0.4,0.010,0.010,green,0.0,GO,light: the evidence very highly suggests that the signal is green | person: evidence very lowly suggests that a person exists
5,0.5,0.142,0.010,green,0.5,GO,light: the evidence very highly suggests that the signal is green | person: evidence very lowly suggests that a person exists
6,0.6,0.121,0.010,green,0.6,GO,light: the evidence very highly suggests that the signal is green | person: evidence very lowly suggests that a person exists
7,0.7,0.010,0.824,green,0.0,STOP,light: the evidence very highly suggests that the signal is green | person: there is evidence that a person exists
8,0.8,0.207,0.777,green,0.0,STOP,light: the evidence very highly suggests that the signal is green | person: there is evidence that a person exists
